# Data Retrieval — Reddit

| Field | Details |
|---|---|
| **Time window** | Jun 2024 – Nov 2024 |
| **Source** | Reddit — JSONL files (Pushshift / Academic Data Access) |
| **Method** | Pre-collected JSONL + cleaning pipeline (deduplication, filtering, normalisation) |
| **Subreddits** | r/conservative · r/democrats · r/republican · r/worldnews · r/politics · r/liberal · r/trump |
| **Volume** | **5,591,506** raw records (posts + comments) → **2,200,841** after cleaning |
| **Saved to** | `Data/1_Bronze/Reddit/*.jsonl` (14 files) · silver: `reddit_clean.parquet` |

### Bronze — 14 JSONL files (comments: ~68 fields · posts: ~95 fields)

Key columns used in analysis:

| Column | Applies to | Description |
|---|---|---|
| `body` | comments | Raw comment text |
| `title` | posts | Post title |
| `selftext` | posts | Post body text |
| `author` | both | Reddit username |
| `subreddit` | both | Subreddit name |
| `created_utc` | both | Unix timestamp of creation |
| `score` | both | Net upvotes |
| `num_comments` | posts | Number of comments on post |
| `parent_id` | comments | ID of parent comment/post |
| `id` | both | Unique Reddit ID |

### Silver — reddit_clean.parquet · 2,200,841 rows × 15 columns

| Column | Description |
|---|---|
| `body` | Original comment text |
| `text_clean` | Cleaned, lowercased text (URLs, markdown, mentions removed) |
| `author` | Reddit username |
| `subreddit` | Subreddit |
| `created_utc` | Creation timestamp |
| `score` | Net upvotes |
| `length_words` | Word count after cleaning |


In [2]:
import pandas as pd

In [8]:
df = pd.read_parquet("C:/Users/seppe/Documents/GitHub/group-project-SMWA/Data/Reddit/reddit_full.parquet")

### 1. Setup

In [9]:
import pandas as pd
import re

# Make a copy of your dataframe
df_clean = df.copy()

print("Initial shape:", df_clean.shape)

Initial shape: (2363964, 14)


### 2. Remove duplicates & missing values

In [10]:
# Remove duplicates
df_clean = df_clean.drop_duplicates(subset=["body", "author", "created_utc"])

# Remove rows with missing values
df_clean = df_clean.dropna(subset=["body", "created_utc"])

print("After removing duplicates & nulls:", df_clean.shape)

After removing duplicates & nulls: (2363048, 14)


### 3. Remove deleted/removed comments

In [11]:
df_clean = df_clean[
    ~df_clean["body"].str.lower().isin(["[deleted]", "[removed]"])
]

print("After removing deleted comments:", df_clean.shape)

After removing deleted comments: (2363048, 14)


### 4. Text cleaning function

In [12]:
def clean_text(text):
    """
    Basic text cleaning for Reddit comments.
    Keeps text usable for NLP but removes noise.
    """
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove markdown links
    text = re.sub(r"\[.*?\]\(.*?\)", "", text)

    # Remove Reddit mentions (u/ and r/)
    text = re.sub(r"u\/\w+", "", text)
    text = re.sub(r"r\/\w+", "", text)

    # Remove HTML entities
    text = re.sub(r"&\w+;", "", text)

    # Remove line breaks and tabs
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

### 5. Apply cleaning

In [13]:
df_clean["text_clean"] = df_clean["body"].apply(clean_text)

print(df_clean[["body", "text_clean"]].head())

                                                body  \
0  from the pic he's looking pretty orange from r...   
1  Nor will they ever, this debacle will serve as...   
2                                  Little California   
3                     Is this a real quote? Oh dear.   
4    He should be proud, quite the accomplishment. 😆   

                                          text_clean  
0  from the pic he's looking pretty orange from r...  
1  nor will they ever, this debacle will serve as...  
2                                  little california  
3                     is this a real quote? oh dear.  
4    he should be proud, quite the accomplishment. 😆  


### 6. Remove empty / low-quality comments

In [14]:
# Remove empty text
df_clean = df_clean[df_clean["text_clean"].str.len() > 10]

# Create word count
df_clean["length_words"] = df_clean["text_clean"].str.split().str.len()

# Remove very short comments
df_clean = df_clean[df_clean["length_words"] > 3]

# Optional: remove extremely long comments
df_clean = df_clean[df_clean["length_words"] < 500]

print("After length filtering:", df_clean.shape)

After length filtering: (2200841, 15)


### 7. Final clean check

In [15]:
print("Final shape:", df_clean.shape)

print("\nComments per subreddit:")
print(df_clean["subreddit"].value_counts())

print("\nPreview:")
print(df_clean[["subreddit", "text_clean"]].head())

Final shape: (2200841, 15)

Comments per subreddit:
subreddit
worldnews       1463456
democrats        330540
conservative     294204
republican       112641
Name: count, dtype: int64

Preview:
      subreddit                                         text_clean
0  conservative  from the pic he's looking pretty orange from r...
1  conservative  nor will they ever, this debacle will serve as...
3  conservative                     is this a real quote? oh dear.
4  conservative    he should be proud, quite the accomplishment. 😆
6  conservative  "noooo, he was actually setting up for desanti...


### 8. Save cleaned data

In [16]:
df_clean.to_parquet("reddit_clean.parquet", index=False)

print("Saved as reddit_clean.parquet")

Saved as reddit_clean.parquet
